In [0]:
bronze_df = spark.table("workspace.default.bronze_transactions")

1.Did we ingest all the records?

In [0]:
%sql
SELECT COUNT(*) AS total_records
FROM workspace.default.bronze_transactions;

total_records
6362620


In [0]:
total_records = bronze_df.count()

print(f"Total Records: {total_records}")

Total Records: 6362620


2.Null Checks

In [0]:
%sql
SELECT
    COUNT(CASE WHEN nameOrig IS NULL THEN 1 END) AS null_source_accounts,
    COUNT(CASE WHEN nameDest IS NULL THEN 1 END) AS null_destination_accounts
FROM workspace.default.bronze_transactions;

null_source_accounts,null_destination_accounts
0,0


In [0]:
from pyspark.sql.functions import col, when, count

bronze_df.select(
    count(when(col("nameOrig").isNull(), 1)).alias("null_source_accounts"),
    count(when(col("nameDest").isNull(), 1)).alias("null_destination_accounts")
).show()

+--------------------+-------------------------+
|null_source_accounts|null_destination_accounts|
+--------------------+-------------------------+
|                   0|                        0|
+--------------------+-------------------------+



3.Negetive amounts

In [0]:
%sql
SELECT COUNT(*) AS negative_amounts
FROM workspace.default.bronze_transactions
WHERE amount < 0;

negative_amounts
0


In [0]:
negative_amounts = bronze_df.filter(col("amount") < 0).count()

print(f"Negative Amounts: {negative_amounts}")

Negative Amounts: 0


4.Duplicate transactions

In [0]:
%sql
SELECT
    step,
    type,
    amount,
    nameOrig,
    nameDest,
    COUNT(*) AS duplicate_count
FROM workspace.default.bronze_transactions
GROUP BY
    step,
    type,
    amount,
    nameOrig,
    nameDest
HAVING COUNT(*) > 1;

step,type,amount,nameOrig,nameDest,duplicate_count


5.Transaction type distribution

In [0]:
%sql
SELECT
    type,
    COUNT(*) AS transaction_count
FROM workspace.default.bronze_transactions
GROUP BY type
ORDER BY transaction_count DESC;

type,transaction_count
CASH_OUT,2237500
PAYMENT,2151495
CASH_IN,1399284
TRANSFER,532909
DEBIT,41432


6.Fraud distribution

In [0]:
%sql
SELECT
    isFraud,
    COUNT(*) AS transaction_count
FROM workspace.default.bronze_transactions
GROUP BY isFraud;

isFraud,transaction_count
1,8213
0,6354407


Building Validation Summary

Record Count 

In [0]:
expected_records = 6362620
actual_records = bronze_df.count()

if actual_records == expected_records:
    print("✅ Record Count Validation: PASS")
else:
    print(f"❌ Record Count Validation: FAIL (Expected: {expected_records}, Actual: {actual_records})")

✅ Record Count Validation: PASS


Null Source accounts

In [0]:
null_source_accounts = bronze_df.filter(
    col("nameOrig").isNull()
).count()

if null_source_accounts == 0:
    print("✅ Source Account Validation: PASS")
else:
    print(f"❌ Source Account Validation: FAIL ({null_source_accounts} null records)")

✅ Source Account Validation: PASS


Negative amounts

In [0]:
negative_amounts = bronze_df.filter(
    col("amount") < 
).count()

if negative_amounts == 0:
    print("✅ Negative Amount Validation: PASS")
else:
    print(f"❌ Negative Amount Validation: FAIL ({negative_amounts} records)")

✅ Negative Amount Validation: PASS


In [0]:
if negative_amounts > 0:
    raise Exception("Pipeline failed: Negative transaction amounts detected.")

Validation result list

In [0]:
validation_results = []

Store Validation Results

In [0]:
if actual_records == expected_records:
    validation_results.append({
        "Validation": "Record Count",
        "Status": "PASS",
        "Actual": actual_records,
        "Expected": expected_records
    })
else:
    validation_results.append({
        "Validation": "Record Count",
        "Status": "FAIL",
        "Actual": actual_records,
        "Expected": expected_records
    })

In [0]:
if negative_amounts == 0:
    validation_results.append({
        "Validation": "Negative Amounts",
        "Status": "PASS",
        "Actual": negative_amounts,
        "Expected": 0
    })
else:
    validation_results.append({
        "Validation": "Negative Amounts",
        "Status": "FAIL",
        "Actual": negative_amounts,
        "Expected": 0
    })

In [0]:
if null_source_accounts == 0:
    validation_results.append({
        "Validation": "Source Account",
        "Status": "PASS",
        "Actual": null_source_accounts,
        "Expected": 0
    })
else:
    validation_results.append({
        "Validation": "Source Account",
        "Status": "FAIL",
        "Actual": null_source_accounts,
        "Expected": 0
    })

Display the Report

In [0]:
report_df = spark.createDataFrame(validation_results)

display(report_df)

Actual,Expected,Status,Validation
6362620,6362620,PASS,Record Count
0,0,PASS,Negative Amounts
0,0,PASS,Source Account
0,0,PASS,Negative Amounts
0,0,PASS,Negative Amounts
0,0,PASS,Source Account
